<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Z-scores, and why the mean is fooled

**[Z-scores, and why the mean is fooled](https://learning.nextia-ai.com/courses/anomaly-detection/m02/z-scores-and-why-the-mean-is-fooled/)** · Anomaly Detection: Find What Is Unusual · Module 2, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure how far a payment is from the usual one with a z-score, and see how one huge payment moves the mean and the standard deviation so much that the next strange payment hides.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | The comparison table, from [Meet the Larkfield data](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/). Mean and standard deviation, from the mathematics course. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m02/z-scores-and-why-the-mean-is-fooled/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M02-L01-z-scores-and-why-the-mean-is-fooled/z-scores-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. How far from usual? A tiny example

Think of a café owner who knows a regular customer: a coffee and a croissant, about 14 dollars, every morning. One morning the bill is 60 dollars. The owner notices at once: it is far from usual.

A **z-score** says "how far from usual" as a number. It is the distance from the **mean** (the average), counted in **standard deviations** (the usual size of the differences from the mean):

z = (value − mean) ÷ standard deviation

A z-score of 0 is exactly usual. Many teams call a value with a z-score above 3 (or below −3) an **outlier**: a value far from the others.

Run the next cell. It computes the z-score of a 60-dollar payment for a customer whose last seven payments were 12 to 16 dollars.

> **Check.** You should see `mean 14.14, standard deviation 1.35, z-score of 60 dollars: 34.1`.

In [ ]:
usual = [12.0, 15.0, 14.0, 13.0, 16.0, 15.0, 14.0]     # one customer's last seven payments, in dollars
new_payment = 60.0

mean = np.mean(usual)
std = np.std(usual, ddof=1)
z = (new_payment - mean) / std
print(f"mean {mean:.2f}, standard deviation {std:.2f}, z-score of {new_payment:.0f} dollars: {z:.1f}")

A z-score of 34.1 is very far from usual. The z-score works.

> **Predict.** Now one huge payment of 900 dollars joins the customer's history (a gift for a wedding, or a stolen card). Then the same 60-dollar payment comes. Is its z-score still above 3?

In [ ]:
with_huge = usual + [900.0]                            # one huge payment joins the history
mean2 = np.mean(with_huge)
std2 = np.std(with_huge, ddof=1)
z2 = (new_payment - mean2) / std2
print(f"mean {mean2:.2f}, standard deviation {std2:.2f}, z-score of {new_payment:.0f} dollars: {z2:.2f}")

> **Check.** You should see a z-score of -0.21.

The 60-dollar payment now looks **normal**: its z-score is below zero. One huge payment moved the mean from 14.14 to 124.88 dollars, and the standard deviation from 1.35 to 313.20. The mean and the standard deviation are **not robust**: one extreme value moves them a lot. This is why the mean is fooled, and it is the problem that the next lesson solves.

## 2. Z-scores on Larkfield's payments

Now the same idea on the real data. The detector learns the mean and the standard deviation from the **reference months** (June to August), and scores every payment of the **evaluation months** (September to November). The team reviews the 50 payments with the highest |z| each day.

> **Check.** You should see `173,271` reference payments, mean 119.36 and standard deviation 223.07.

In [ ]:
import compare
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
mean, std = reference.amount.mean(), reference.amount.std()
print(f"reference months: {len(reference):,} payments, mean {mean:.2f}, standard deviation {std:.2f}")
print(f"evaluation months: {len(evaluation):,} payments")

Run the next cell. It scores every evaluation payment with its |z| (the size of z, without the sign) and adds the first row of this module's comparison table.

> **Check.** You should see a precision at budget of 0.003 and a recall of 0.005.

In [ ]:
z_global = ((evaluation.amount - mean) / std).abs()
table = compare.Table()
table.add(compare.score_payments(evaluation, z_global, "z-score", "|z| of the amount, one baseline for all",
                                 explain_minutes=1))
table.show()

Almost nothing. The largest amounts of the shop are business customers' orders, and most frauds are small payments. Run the next cell to see where z = 3 is.

In [ ]:
fig, ax = plt.subplots()
ax.hist(np.log10(evaluation.amount), bins=80)
ax.axvline(np.log10(mean + 3 * std), color="red", label="z = 3")
ax.set_xlabel("amount, log10 of dollars (1 = 10 dollars, 2 = 100, 3 = 1,000)")
ax.set_ylabel("payments")
ax.legend()
plt.show()
print(f"z = 3 is an amount of {mean + 3 * std:,.0f} dollars")

Most payments are between 10 and 300 dollars. One baseline for every customer cannot see a small payment that is strange **for this customer**.

## 3. Predict, then run: one baseline per customer

Each customer gets their own mean and standard deviation, from their own reference payments. A customer with fewer than 5 reference payments (new customers, for example) keeps the shop's baseline.

> **Predict.** Will the per-customer z-score find more planted frauds than the global one? Write down your guess for its precision at budget.

In [ ]:
stats = reference.groupby("customer_id").amount.agg(["count", "mean", "std"])
known = evaluation.customer_id.map(stats["count"]).fillna(0) >= 5      # at least 5 payments to learn from
c_mean = evaluation.customer_id.map(stats["mean"]).where(known, mean)
c_std = evaluation.customer_id.map(stats["std"]).where(known, std)
z_customer = ((evaluation.amount - c_mean) / c_std).abs()
row = table.add(compare.score_payments(evaluation, z_customer, "z-score per customer",
                                       "|z| against the customer's own mean (5+ payments)", explain_minutes=1))
table.show()
print("customers with 5 or more reference payments:", int((stats["count"] >= 5).sum()))

> **Check.** You should see a precision at budget of 0.045 for the per-customer z-score, against 0.003 for the global one.

Better, but still low. Two reasons, which the next lessons fix: one past huge payment fools a customer's own mean too (you saw it in section 1), and many frauds are not about the amount at all.

## 4. Change one thing: a fixed limit instead of the budget

The team can review 50 alerts a day. What if the detector sends every payment with |z| > 3 instead of the top 50? Run the next cell.

In [ ]:
flags = (z_global > 3).astype(int)
row3 = compare.score_payments(evaluation, flags, "z-score above 3", "every payment with |z| > 3", threshold=1,
                              explain_minutes=1)
print(f"alerts per day with |z| > 3: {row3['alerts per day']}, precision of the 50 reviewed: {row3['precision at budget']}")

> **Check.** You should see 33.9 alerts per day.

"|z| > 3" is a habit, not a law. Here it would send about 34 alerts a day, fewer than the team can review. The precision does not change (0.003): the alerts are still the largest amounts. A limit decides **how many** alerts you get, not **which** frauds you find. In Module 7, you set the threshold from the budget on purpose.

## 5. A common mistake: a spread of zero

Some customers always pay the same amount. Their standard deviation is 0, and the z-score divides by zero. Run the next cell.

In [ ]:
same = [20.0, 20.0, 20.0, 20.0, 20.0]                   # a customer who always pays the same amount
z_bad = (25.0 - np.mean(same)) / np.std(same, ddof=1)
print("z-score of 25 dollars:", z_bad)

NumPy prints a warning and gives `inf` (infinity). In pandas, a whole column of z-scores then has `inf` values, which sort first and fill the budget with nothing interesting. The fix: a floor for the spread. Run the next cell.

In [ ]:
z_fixed = (25.0 - np.mean(same)) / max(np.std(same, ddof=1), 1.0)   # a floor of 1 dollar for the spread
print("z-score of 25 dollars, with a floor:", z_fixed)

## 6. Check yourself

**Your turn 1.** In section 1, what is the z-score of the 60-dollar payment **after** the huge payment? Write it as `z_after` (with 2 decimals).

In [ ]:
z_after = -0.21

In [ ]:
expect('z_after', z_after, -0.21, tolerance=0.01)

**Your turn 2.** With the global z-score, how many evaluation payments a day have |z| above 4? Write the number of alerts per day as `alerts_per_day_above_4`, rounded to 1 decimal.

In [ ]:
alerts_per_day_above_4 = round(float((z_global > 4).sum() / evaluation.ts.dt.floor("D").nunique()), 1)

In [ ]:
expect('alerts_per_day_above_4', alerts_per_day_above_4, 18.0, tolerance=0.001)

**Your turn 3.** A fraud is a payment **above** the customer's usual amount more often than below it. Score only the high side: keep the sign of the per-customer z (`(evaluation.amount - c_mean) / c_std`), and set every negative value to 0. Keep the row as `up_row`. Is it better than |z|?

In [ ]:
first_z = (evaluation.amount - c_mean) / c_std
up_only = first_z.clip(lower=0)            # only amounts ABOVE the customer's usual one
up_row = compare.score_payments(evaluation, up_only, "z-score per customer, high side", "only amounts above usual")

In [ ]:
expect_hash('the precision of the high-side z-score', up_row['precision at budget'] if isinstance(up_row, dict) else None, '433e2c64ec59fcfc')

The high side alone gives 0.024. Many planted frauds are **small** payments (stolen cards, card testing), so they are below the customer's usual amount: an amount-only detector misses them, whatever the side.

## Recap

- A z-score is the distance from the mean in standard deviations. It is easy to compute and to explain.
- The mean and the standard deviation are not robust: one huge payment hid the next strange one (34.1 became -0.21).
- One baseline for everybody found almost nothing; one per customer did better, and still missed the frauds that are not about the amount.

The next lesson uses the median and the MAD, which one huge payment does not move.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m02/z-scores-and-why-the-mean-is-fooled/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.